In [1]:
#import imodex_miri_postprocess_pipeline as imp #homemade pipeline

# Basic system utilities for interacting with files
import glob, os 
from collections import defaultdict
import numpy as np
from astropy.io import fits

# for JWST calibration pipeline
import jwst
import crds
from jwst.pipeline import Detector1Pipeline
from jwst.pipeline import Image2Pipeline

# Echo pipeline version and CRDS context in use
print(f"JWST Calibration Pipeline Version: {jwst.__version__}")
print(f"Using CRDS Context: {crds.get_context_name('jwst')}")

/home/cperrot/anaconda3/envs/jwst/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JWST Calibration Pipeline Version: 3.0.0


CRDS - INFO -  Calibration SW Found: jwst 3.0.0 (/home/cperrot/anaconda3/envs/jwst/lib/python3.13/site-packages/jwst-3.0.0.dist-info)


Using CRDS Context: jwst_1584.pmap


In [4]:
#### STAGE 1 ###

def stage1(files, output_dir):
    det1dict = defaultdict(dict)
    det1dict['jump']['maximum_cores'] = 'half' # Turn on multi-core processing (off by default).  Choose what fraction of cores to use (quarter, half, or all)
    for f in files:
        Detector1Pipeline.call(f, output_dir=output_dir, steps=det1dict, save_results=True)


def MIRI_stage_1(uncal_dir, rate_dir, filter_name, num_obs_sci, num_obs_ref, num_obs_bkgsci, num_obs_bkgref):
    ### Create folder for output and proceed to the stage 1 data reduction
    ### Input example :
    # uncal_dir      = "/home/login/data/JWST/MIRI/UNCAL/P03662/       # String - Path where UNCAL (Stage 0 data) observation are
    # rate_dir       = "/home/login/data/JWST/MIRI/RATE/P03662/        # String - Path where RATE (Stage 1 data) observation are (saved)
    # filter_name    = ["F1140C"] / ["F1065C","F1140C"]                # Array of String - Name of filter used
    # num_obs_sci    = [["001","002"]] / [["007","012"],["009","011"]] # Array of Array of String - Obs number of the Science acquisition for each filters
    # num_obs_ref    = [["004"]] / [["002"],[004"]]                    # Array of Array of String - Obs number of the Reference acquisition for each filters
    # num_obs_bkgsci = [["003"]] / [["008"],["010"]]                   # Array of Array of String - Obs number of the Background Science acquisition for each filters
    # num_obs_bkgref = [["005"]] / [["003"],["005"]]                   # Array of Array of String - Obs number of the Background Reference acquisition for each filters
    ###

    assert os.path.exists(uncal_dir), "Wrong rate directory !"

    if not os.path.exists(rate_dir): os.makedirs(rate_dir)
    
    # Loot on filters
    nb_filter = len(filter_name)
    for ft in range(nb_filter):
                     
        #Coronagraphic Science obs + TA
        for id_s in range(len(num_obs_sci[ft])):
            if not os.path.exists(rate_dir+"obs"+str(num_obs_sci[ft][id_s])+"/"):    os.makedirs(rate_dir+"obs"+str(num_obs_sci[ft][id_s])+"/")
            if not os.path.exists(rate_dir+"obs"+str(num_obs_sci[ft][id_s])+"/TA/"): os.makedirs(rate_dir+"obs"+str(num_obs_sci[ft][id_s])+"/TA/")
            
            #Stage 1 files for Science and associate TA
            files_sci    = sorted(glob.glob(uncal_dir+'obs'+str(num_obs_sci[ft][id_s])+'/*_mirimage_uncal.fits'))
            files_sci_TA = sorted(glob.glob(uncal_dir+'obs'+str(num_obs_sci[ft][id_s])+'/TA/*_mirimage_uncal.fits'))
            stage1(files_sci,    rate_dir+"obs"+str(num_obs_sci[ft][id_s])+"/")
            stage1(files_sci_TA, rate_dir+"obs"+str(num_obs_sci[ft][id_s])+"/TA/")            
            
        #Coronagraphic Reference obs + TA
        for id_r in range(len(num_obs_ref[ft])):
            if not os.path.exists(rate_dir+"obs"+str(num_obs_ref[ft][id_r])+"/"):    os.makedirs(rate_dir+"obs"+str(num_obs_ref[ft][id_r])+"/")
            if not os.path.exists(rate_dir+"obs"+str(num_obs_ref[ft][id_r])+"/TA/"): os.makedirs(rate_dir+"obs"+str(num_obs_ref[ft][id_r])+"/TA/")    
            
            #Stage 1 files for Reference and associate TA
            files_ref    = sorted(glob.glob(uncal_dir+'obs'+str(num_obs_ref[ft][id_r])+'/*_mirimage_uncal.fits'))
            files_ref_TA = sorted(glob.glob(uncal_dir+'obs'+str(num_obs_ref[ft][id_r])+'/TA/*_mirimage_uncal.fits'))
            stage1(files_ref,    rate_dir+"obs"+str(num_obs_ref[ft][id_r])+"/")
            stage1(files_ref_TA, rate_dir+"obs"+str(num_obs_ref[ft][id_r])+"/TA/") 
                      
        # Background Science obs            
        for id_bs in range(len(num_obs_bkgsci[ft])):
            if not os.path.exists(rate_dir+"obs"+str(num_obs_bkgsci[ft][id_bs])+"/"): os.makedirs(rate_dir+"obs"+str(num_obs_bkgsci[ft][id_bs])+"/")
                
            files_bkg = sorted(glob.glob(uncal_dir+'obs'+str(num_obs_bkgsci[ft][id_bs])+'/*_mirimage_uncal.fits'))
            stage1(files_bkg, rate_dir+"obs"+str(num_obs_bkgsci[ft][id_bs])+"/") 
             
        # Background Reference obs    
        for id_br in range(len(num_obs_bkgref[ft])):
            if not os.path.exists(rate_dir+"obs"+str(num_obs_bkgref[ft][id_br])+"/"): os.makedirs(rate_dir+"obs"+str(num_obs_bkgref[ft][id_br])+"/")
                
            files_ref_bkg = sorted(glob.glob(uncal_dir+'obs'+str(num_obs_bkgref[ft][id_br])+'/*_mirimage_uncal.fits')) 
            stage1(files_ref_bkg, rate_dir+"obs"+str(num_obs_bkgref[ft][id_br])+"/") 


### STAGE 2 ###

def stage2(files, output_dir, output_file):
    image2dict = defaultdict(dict) # Set up a dictionary to define how the Image2 pipeline should be configured.
    image2dict['bkg_subtract']['skip'] = True
    image2dict['flat_field']['skip'] = True 
    image2dict['photom']['skip'] = False
    image2dict['assign_wcs']['skip'] = False

    incr = 1 #for the dither number
    for f in files:
        Image2Pipeline.call(f, output_dir=output_dir, output_file=output_file+'_dither'+str(incr), steps=image2dict, save_results=True)
        incr += 1

def MIRI_stage_2_no_bkg(rate_dir, cal_dir, prog_id, name_sci, name_ref, filter_name, num_obs_sci, num_obs_ref, num_obs_bkgsci, num_obs_bkgref):
    ### Create folder for output and proceed to the stage 2 data reduction
    ### Input example :
    # rate_dir       = "/home/login/data/JWST/MIRI/RATE/P03662/        # String - Path where RATE (Stage 1 data) observation are (saved)
    # cal_dir        = "/home/login/data/JWST/MIRI/CAL/P03662/         # String - Path where CAL (Stage 2 data) observation are (saved)
    # prog_id        = 'P03662'                                        # String - ID of the program (format = Pxxxx)
    # name_sci       = 'TWA7'                                          # String - name of the target
    # name_ref       = 'REF_CD23_9765'                                 # String - name of the reference star
    # filter_name    = ["F1140C"] / ["F1065C","F1140C"]                # Array of String - Name of filter used
    # num_obs_sci    = [["001","002"]] / [["007","012"],["009","011"]] # Array of Array of String - Obs number of the Science acquisition for each filters
    # num_obs_ref    = [["004"]] / [["002"],[004"]]                    # Array of Array of String - Obs number of the Reference acquisition for each filters
    # num_obs_bkgsci = [["003"]] / [["008"],["010"]]                   # Array of Array of String - Obs number of the Background Science acquisition for each filters
    # num_obs_bkgref = [["005"]] / [["003"],["005"]]                   # Array of Array of String - Obs number of the Background Reference acquisition for each filters
    ###
    
    assert os.path.exists(rate_dir), "Wrong rate directory !"

    if not os.path.exists(cal_dir): os.makedirs(cal_dir)
    
    # Loot on filters
    nb_filter = len(filter_name)
    for ft in range(nb_filter):
                     
        #Coronagraphic Science obs + TA
        for id_s in range(len(num_obs_sci[ft])):
            if not os.path.exists(cal_dir+"obs"+str(num_obs_sci[ft][id_s])+"/"):    os.makedirs(cal_dir+"obs"+str(num_obs_sci[ft][id_s])+"/")
            if not os.path.exists(cal_dir+"obs"+str(num_obs_sci[ft][id_s])+"/TA/"): os.makedirs(cal_dir+"obs"+str(num_obs_sci[ft][id_s])+"/TA/")
            
            #Stage 1 files for Science and associate TA
            files_sci    = sorted(glob.glob(rate_dir+'obs'+str(num_obs_sci[ft][id_s])+'/*_mirimage_rate.fits'))
            files_sci_TA = sorted(glob.glob(rate_dir+'obs'+str(num_obs_sci[ft][id_s])+'/TA/*_mirimage_rate.fits'))
            stage2(files_sci,    cal_dir+"obs"+str(num_obs_sci[ft][id_s])+"/",    prog_id+'_'+str(num_obs_sci[ft][id_s])+'_'+name_sci)
            stage2(files_sci_TA, cal_dir+"obs"+str(num_obs_sci[ft][id_s])+"/TA/", prog_id+'_'+str(num_obs_sci[ft][id_s])+'_'+name_sci+'_TA')            
            
        #Coronagraphic Reference obs + TA
        for id_r in range(len(num_obs_ref[ft])):
            if not os.path.exists(cal_dir+"obs"+str(num_obs_ref[ft][id_r])+"/"):    os.makedirs(cal_dir+"obs"+str(num_obs_ref[ft][id_r])+"/")
            if not os.path.exists(cal_dir+"obs"+str(num_obs_ref[ft][id_r])+"/TA/"): os.makedirs(cal_dir+"obs"+str(num_obs_ref[ft][id_r])+"/TA/")    
            
            #Stage 1 files for Reference and associate TA
            files_ref    = sorted(glob.glob(rate_dir+'obs'+str(num_obs_ref[ft][id_r])+'/*_mirimage_rate.fits'))
            files_ref_TA = sorted(glob.glob(rate_dir+'obs'+str(num_obs_ref[ft][id_r])+'/TA/*_mirimage_rate.fits'))
            stage2(files_ref,    cal_dir+"obs"+str(num_obs_ref[ft][id_r])+"/",    prog_id+'_'+str(num_obs_ref[ft][id_r])+'_'+name_ref)
            stage2(files_ref_TA, cal_dir+"obs"+str(num_obs_ref[ft][id_r])+"/TA/", prog_id+'_'+str(num_obs_ref[ft][id_r])+'_'+name_ref+'_TA') 
                      
        # Background Science obs            
        for id_bs in range(len(num_obs_bkgsci[ft])):
            if not os.path.exists(cal_dir+"obs"+str(num_obs_bkgsci[ft][id_bs])+"/"): os.makedirs(cal_dir+"obs"+str(num_obs_bkgsci[ft][id_bs])+"/")
                
            files_bkg = sorted(glob.glob(rate_dir+'obs'+str(num_obs_bkgsci[ft][id_bs])+'/*_mirimage_rate.fits'))
            stage2(files_bkg, cal_dir+"obs"+str(num_obs_bkgsci[ft][id_bs])+"/", prog_id+'_'+str(num_obs_bkgsci[ft][id_bs])+'_'+name_sci+'_background') 
             
        # Background Reference obs    
        for id_br in range(len(num_obs_bkgref[ft])):
            if not os.path.exists(cal_dir+"obs"+str(num_obs_bkgref[ft][id_br])+"/"): os.makedirs(cal_dir+"obs"+str(num_obs_bkgref[ft][id_br])+"/")
                
            files_ref_bkg = sorted(glob.glob(rate_dir+'obs'+str(num_obs_bkgref[ft][id_br])+'/*_mirimage_rate.fits')) 
            stage2(files_ref_bkg, cal_dir+"obs"+str(num_obs_bkgref[ft][id_br])+"/", prog_id+'_'+str(num_obs_bkgref[ft][id_br])+'_'+name_ref+'_background') 


### MASTER BACKGROUND ###

def MIRI_stage_2_master_background(cal_dir, master_bkg_dir, prog_id, name, num_obs, method="minimum"):
    ### Do the minimum or mean between dither 1 and dither 2 of a background observation
    # cal_dir        = "/home/login/data/JWST/MIRI/CAL/P03662/         # String - Path where CAL (Stage 2 data) observation are (saved)
    # master_bkg_dir = "/home/login/data/JWST/MIRI/MASTER_BKG/P03662/  # String - Path where Master Background (Stage 2 data) observation are (saved)
    # prog_id        = 'P03662'                                        # String - ID of the program (format = Pxxxx)
    # num_obs        = [["003"],["005"]]                               # Array of Array of String - Obs number of the Background
    # method         = "minimum" or "mean"                             # String - Method used
    ###
    
    if (method != "minimum") and (method != "mean"):
        method = "minimum"
    print('Master background method : '+method)

    nb_obs = len(num_obs)

    if not os.path.exists(master_bkg_dir): os.makedirs(master_bkg_dir)

    #loop on obs 
    for ft in range(nb_obs):
        if not os.path.exists(master_bkg_dir+"obs"+str(num_obs[ft])+"/"): os.makedirs(master_bkg_dir+"obs"+str(num_obs[ft])+"/")                
        files = sorted(glob.glob(cal_dir+'obs'+str(num_obs[ft])+'/'+prog_id+'_'+str(num_obs[ft])+'_'+name+'_background_dither*_cal.fits'))
        img = fits.open(files[0])
        
        if len(files) == 1:
            print("Only one dither.")
            master_bkg = fits.open(files[0])[1].data
        else:
            dither1 = fits.open(files[0])[1].data
            dither2 = fits.open(files[1])[1].data
            if method == "minimum":
                master_bkg = np.minimum(dither1,dither2)
            elif method == "mean":
                master_bkg = np.nanmean([dither1,dither2], axis=0)

        mbkg_name = prog_id+'_'+str(num_obs[ft])+'_'+name+'_master_bkg_cal.fits'
        img[0].header['FILENAME'] = mbkg_name
        hdu = fits.PrimaryHDU()
        hdu.data = master_bkg
        hdu.header = img[0].header
        hdu.writeto(master_bkg_dir+'obs'+str(num_obs[ft])+'/'+mbkg_name,overwrite=True)
        img.close()
        print("master background saved : "+master_bkg_dir+'obs'+str(num_obs[ft])+'/'+prog_id+'_'+str(num_obs[ft])+'_'+name+'_master_bkg_cal.fits')



        

### Batch example below

In [5]:
### P07340 : GO - A NIRCam - MIRI - ALMA synergy to constrain planet formation at large orbital separations      
# AS209 

dostage1       = False  # calwebb_detector1 from uncal to rate
dostage2       = True   # calwebb_image2 from rate to cal
domasterbkg    = True   # home made, do master background from both bkg dither (minimum or mean)

prog_id        = 'P07340'

uncal_dir      = "/home/cperrot/Work/raw/JWST/Target/MAST/UNCAL/"+prog_id+"/"
rate_dir       = "/home/cperrot/Work/raw/JWST/Target/MAST/RATE/"+prog_id+"/"
cal_dir        = "/home/cperrot/Work/raw/JWST/Target/MAST/CAL/"+prog_id+"/"
master_bkg_dir = "/home/cperrot/Work/raw/JWST/Target/MAST/MASTER_BKG/"+prog_id+"/"

name_sci       = 'AS209'
name_ref       = 'REF_24Sco'
filter_name    = ["F1140C"]
# For each filter : Science, Reference, Sci Background, Ref Background
num_obs_sci    = [["001","002"]] # Sciences Roll 
num_obs_ref    = [["003"]] # Reference Roll 
num_obs_bkgsci = [["004"]] # Science Background
num_obs_bkgref = [["005"]] # Reference Background

if dostage1:
    MIRI_stage_1(uncal_dir, rate_dir, filter_name, num_obs_sci, num_obs_ref, num_obs_bkgsci, num_obs_bkgref)

if dostage2:
    MIRI_stage_2_no_bkg(rate_dir, cal_dir, prog_id, name_sci, name_ref, filter_name, num_obs_sci, num_obs_ref, num_obs_bkgsci, num_obs_bkgref)

if domasterbkg:
    MIRI_stage_2_master_background(cal_dir, master_bkg_dir, prog_id, name_sci, num_obs_bkgsci[0], method="minimum")
    MIRI_stage_2_master_background(cal_dir, master_bkg_dir, prog_id, name_ref, num_obs_bkgref[0], method="minimum")






2026-10-28 13:59:37,017 - stpipe.step - INFO - PARS-RESAMPLESTEP parameters found: /home/cperrot/crds_cache/references/jwst/miri/jwst_miri_pars-resamplestep_0001.asdf


2026-10-28 13:59:37,030 - stpipe.step - INFO - Image2Pipeline instance created.
2026-10-28 13:59:37,031 - stpipe.step - INFO - BackgroundStep instance created.
2026-10-28 13:59:37,032 - stpipe.step - INFO - AssignWcsStep instance created.
2026-10-28 13:59:37,033 - stpipe.step - INFO - FlatFieldStep instance created.
2026-10-28 13:59:37,033 - stpipe.step - INFO - PhotomStep instance created.
2026-10-28 13:59:37,035 - stpipe.step - INFO - ResampleStep instance created.
2026-10-28 13:59:37,141 - stpipe.step - INFO - Step Image2Pipeline running with args ('/home/cperrot/Work/raw/JWST/Target/MAST/RATE/P07340/obs001/jw07340001001_04101_00001_mirimage_rate.fits',).
2026-10-28 13:59:37,146 - stpipe.step - INFO - Step Image2Pipeline parameters are:
  pre_hooks: []
  post_hooks: []
  output_file: /home/cperrot/Work/code/JWST/Target/GTO/P07340_001_AS209_dither1
  output_dir: /home/cperrot/Work/raw/JWST/Target/MAST/CAL/P07340/obs001/
  output_ext: .fits
  output_use_model: False
  output_use_index

Master background method : minimum
master background saved : /home/cperrot/Work/raw/JWST/Target/MAST/MASTER_BKG/P07340/obs004/P07340_004_AS209_master_bkg_cal.fits
Master background method : minimum
master background saved : /home/cperrot/Work/raw/JWST/Target/MAST/MASTER_BKG/P07340/obs005/P07340_005_REF_24Sco_master_bkg_cal.fits
